In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [4]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/stereo/blos/fdt/*'))
files_fdt

['/home/ulyanov/data/stereo/blos/fdt/solo_L2_phi-fdt-bamb_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/blos/fdt/solo_L2_phi-fdt-bazi_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/blos/fdt/solo_L2_phi-fdt-binc_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/blos/fdt/solo_L2_phi-fdt-blos_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/blos/fdt/solo_L2_phi-fdt-blos_20251007T124003_V202602220437_0550070502.fits.gz']

In [4]:
file_fdt = files_fdt[3]

for i in range(0, 200 + 1):

    with fits.open(file_fdt) as hdul:
        header_fdt = hdul[0].header
        data_fdt = hdul[0].data


    did = int(file_fdt.split('_')[-1].split('.')[0])
    data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1.25

    view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                                   crota=header_fdt['CROTA'] + 0.25)

    q = np.sin(i / 100 * np.pi) * 44.9
    view_new = view_fdt.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=q * 3600)#, crln=view_fdt.crln+180)
    transform_fdt = (view_new.to_carrington() -
                     view_fdt.to_carrington(mu_thr=0.1))
    grid, alpha = transform_fdt(view_new.grid())
    data_fdt = bilinear(data_fdt, *grid) * alpha

    show_data(data_fdt, view_new, label=r'$B_{los}, G$', to_file=f'temp/{i:03d}.png')

In [17]:
file_fdt = files_fdt[4]

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data


did = int(file_fdt.split('_')[-1].split('.')[0])
data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1.25

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                               crota=header_fdt['CROTA'] + 0.25)
view_new = view_fdt.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=-25 * 3600)#, crln=view_fdt.crln+180)
transform_fdt = (view_new.to_carrington() -
                 view_fdt.to_carrington(mu_thr=0.1))
grid, alpha = transform_fdt(view_new.grid())
data_fdt = bilinear(data_fdt, *grid) * alpha

show_data(data_fdt, view_new, 'FDT', label=r'$B_{los}, G$')

In [6]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/stereo/blos/hmi/*'))
files_hmi

['/home/ulyanov/data/stereo/blos/hmi/hmi.M_45s.20250924_062015_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/blos/hmi/hmi.M_45s.20251007_124330_TAI.2.magnetogram.fits']

In [18]:
file_hmi = files_hmi[1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data


data_hmi = rebin(data_hmi, 4, update_header=header_hmi)
view_hmi = View.from_header(header_hmi)
view_new_ = view_new.update(rsun_arc=-25 * 3600)#, crlt=view_hmi.crlt, crln=view_hmi.crln)
transform_hmi = (view_new_.to_carrington() -
                 view_hmi.to_carrington(mu_thr=0.1))

grid, alpha = transform_hmi(view_new_.grid())
data_hmi = bilinear(data_hmi, *grid) * alpha

show_data(data_hmi, view_new_, 'HMI', label=r'$B_{los}, G$')

In [ ]:
transform = view_hmi.to_carrington(origin='helioprojective') - view_fdt.to_carrington(origin='helioprojective')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)


delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt.copy()
v2 = data_hmi.copy()

#v1[np.abs(v1) < 15] = np.nan
v2[np.abs(v2) < 15] = np.nan

v2_ = (v2 - v1 * q) / delta


print(e2_, np.arccos(q) * 180 / np.pi)

In [ ]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(np.abs(v1 * zi + v2_ * (e2_[0] * xi + e2_[1] * yi)) / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=0, vmax=1)
plt.tight_layout()

In [ ]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(v1 * zi / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=-1, vmax=1)
plt.tight_layout()

In [ ]:
with fits.open(files_fdt[2]) as hdul:
    header_inc = hdul[0].header
    data_inc = hdul[0].data

data_inc = undistort(data_inc, header_inc, xd, yd)
grid, alpha = transform_fdt(view_new.grid())
data_inc = bilinear(data_inc, *grid) * alpha

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(np.cos(data_inc * np.pi / 180), 'jet', vmin=-1, vmax=1)
plt.tight_layout()

In [ ]:
with fits.open(files_fdt[1]) as hdul:
    header_azi = hdul[0].header
    data_azi = hdul[0].data

data_azi = undistort(data_azi, header_azi, xd, yd)
grid, alpha = transform_fdt(view_new.grid())
data_azi = bilinear(data_azi, *grid) * alpha

crota = header_azi['CROTA'] + 0.25
data_azi += crota

In [ ]:
azi = (data_azi + ((np.sin(data_azi * np.pi / 180) * v2_) > 0) * 180) % 360
azi[np.isnan(v2)] = np.nan

plt.figure(figsize=(10,10))
plt.imshow(azi, 'hsv', vmin=0, vmax=360, origin='lower')
plt.tight_layout()

In [ ]:
with fits.open(files_fdt[0]) as hdul:
    header_amb = hdul[0].header
    data_amb = hdul[0].data

#data_amb = undistort(data_amb, header_amb, xd, yd)
#grid, alpha = transform_fdt(view_new.grid())
#data_amb = bilinear(data_amb, *grid) * alpha

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow((np.sin((data_azi + crota) * np.pi / 180) * v2_) > 0)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(data_amb[0])
plt.tight_layout()